In [1]:
import numpy as np
import torch

import os
from tqdm import tqdm


In [2]:
os.environ["CUDA_VISIBLE_DEVICES"]=str(6)

In [3]:
train_data='gsv'
method='boq'
dataset='nordland'
lr=1e-7
num="" 
patience=None
saved_path_log= f"../logs/{train_data}_{method}/logfile{train_data}_{method}_finetuned_on_{dataset}_{lr}_{num}.pth"
saved_path_model= f"../logs/best_model_{train_data}_{method}/{train_data}_{method}_finetuned_on_{dataset}_{lr}_{num}.pth"
# saved_path= "logs/gsv_crica/logfilegsv_crica_finetuned_on_nordland.pth1e-05"
logfile=torch.load(saved_path_log)
args=logfile['args']
original_model=True

In [9]:
method="salad"

if method=="boq":
    image_size=[322,322]
    descriptors_dimension = 12288

        
    model = torch.hub.load("amaralibey/bag-of-queries", "get_trained_boq", backbone_name="dinov2", output_dim=12288)
#     for param in model.backbone.named_parameters():
#         if param[1].requires_grad:
#             param[1].requires_grad=False

    
elif method == "crica":
    image_size=[224,224]
    descriptors_dimension = 10752
    model=torch.hub.load("Lu-Feng/CricaVPR", "trained_model")
    args.features_dim=descriptors_dimension
elif method == "salad":
    image_size=[322,322]
    descriptors_dimension=8448
    features_dim=descriptors_dimension
    model = torch.hub.load("serizba/salad", "dinov2_salad")
    
    
if not original_model:
    state_dict=torch.load(saved_path_model)["model_state_dict"]
    model.load_state_dict()

Using cache found in /home/osverburg/.cache/torch/hub/serizba_salad_main
Using cache found in /home/osverburg/.cache/torch/hub/facebookresearch_dinov2_main


In [13]:
wanted_dataset1="gsvcities"
wanted_dataset2="amstertime"

In [10]:
y=0
x=0
backbone_num=0
backbone_trainable=0
agg_trainable=0
agg_num=0
if method == "crica":
    for name, param in model.module.backbone.named_parameters():
        if "adapter" not in name:
            param.requires_grad = False
if method == "salad":
    for blk in model.backbone.model.blocks[:-model.backbone.num_trainable_blocks]:
    #     print(blk)
        for param in blk.parameters():
            param.requires_grad=False
model.train()
# print(model.module.backbone.blocks[5].attn)

for param in model.named_parameters():
#     print(param[0])
    y+=param[1].numel()
    if "backbone" in param[0]:
        backbone_num+=param[1].numel()
    if "backbone" not in param[0]:
        agg_num+=param[1].numel()
    if param[1].requires_grad and "backbone" in param[0]:
        backbone_trainable+=param[1].numel()
    
    if param[1].requires_grad and "backbone" not in param[0]:
        agg_trainable+=param[1].numel()
    
    if param[1].requires_grad:
#         print(param[0])
        x+=param[1].numel()


print(f"model: {method} with {y:.3e} parameters is trainable on {x:.3e} parameters the backbone has {backbone_num:.3e} parameter and {backbone_trainable:.3e} are trainable")
print(f"aggregator has {agg_num:.3e} parameter and is trainable on {agg_trainable:.3e}")

model: salad with 8.799e+07 parameters is trainable on 3.128e+07 parameters the backbone has 8.658e+07 parameter and 2.987e+07 are trainable
aggregator has 1.411e+06 parameter and is trainable on 1.411e+06
